<a href="https://colab.research.google.com/github/TeodoraV7/hip-sling-market-analysis/blob/survey_transformations/notebooks/survey_transfromation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installation of non-default packages
!pip install geopy -q

# Import libraries
import re
import pandas as pd
import time

from datetime import datetime, timezone
from google.colab import auth
from google.cloud import bigquery
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [ ]:
#Authenticate with  BigQuery

auth.authenticate_user()

PROJECT_ID = "nomad-503212"
client = bigquery.Client(project=PROJECT_ID)

In [ ]:
# Load raw survey results

DATASET = "staging"
TABLE = "stg_survey_responses"

query = f"""
    SELECT * EXCEPT (Your_email_address)
    FROM `{PROJECT_ID}.{DATASET}.{TABLE}`
"""

df_raw = client.query(query).to_dataframe()

In [ ]:
RENAME_MAP = {
    'Timestamp': 'timestamp',
    'Your_child_s_age___If_you_have_more_than_one_child__only_select_the_details_for_the_one_who_is_the_youngest_and_above_the_age_of_1': 'child_age',
    'Your_child_weight__kg_____If_you_have_more_than_one_child__only_select_the_details_for_the_one_who_is_the_youngest_and_above_the_age_of_1': 'child_weight_kg',
    'How_many_children_do_you_have_': 'num_children',
    'In_what_area_does_your_family_live_': 'area_type',
    'How_often_do_you_spend_time_outdoors_with_your_child__walks__park__beach_____': 'outdoor_frequency',
    'What_types_of_activities_do_you_do_most_often_as_a_family____Choose_up_to_3': 'family_activities',
    'What_type_of_transport_do_you_use_most_frequently_in_your_daily_life____Choose_up_to_3': 'transport_type',
    'How_much_time_do_you_usually_spend_outside_with_your_child_on_a_typical_day_': 'outdoor_time_per_day_hrs',
    'Do_you_currently_use_a_baby_carrier__': 'uses_carrier_current',
    'How_often_do_you_use_a_baby_carrier_': 'carrier_usage_frequency',
    'What_type_of_baby_carrier_do_you_use_': 'carrier_type_used',
    'If_you_use_a_carrier__what_is_the_brand_and_model____Example__BabyBjorn__Harmony': 'carrier_brand_model',
    'Do_yo_currently_use_hip_sling_carrier_': 'uses_hip_sling_current',
    'What_would_make_you_switch_to_a_new_hip_sling_carrier_like_the_one_on_the_picture_____Choose_up_to_5': 'switch_reasons',
    'Your_email_address': 'email',
    'In_which_city_and_country_are_you_based___Please_answer_in_english__example__Madrid__Spain_': 'location_raw',
    'How_old_are_you_': 'respondent_age',
    'What_is_your_gender': 'gender',
    'How_would_you_rate_your_satisfaction_with_your_current_carrier___1__not_at_all_satisfied__5__very_satisfied': 'satisfaction_overall',
    'How_satisfied_are_you_with_your_current_carrier_in_terms_of__Panel_material_': 'satisfaction_panel_material',
    'How_satisfied_are_you_with_your_current_carrier_in_terms_of__Other_materials__zips__straps_etc___': 'satisfaction_other_materials',
    'How_satisfied_are_you_with_your_current_carrier_in_terms_of__How_cofortable_it_is_for_you_': 'satisfaction_comfort_parent',
    'How_satisfied_are_you_with_your_current_carrier_in_terms_of__How_comfortable_it_is_for_your_child_': 'satisfaction_comfort_child',
    'How_satisfied_are_you_with_your_current_carrier_in_terms_of__Durability___build_quality_': 'satisfaction_durability',
    'How_satisfied_are_you_with_your_current_carrier_in_terms_of__Ease_of_putting_on_taking_off_': 'satisfaction_ease_of_use',
    'How_satisfied_are_you_with_your_current_carrier_in_terms_of__Compactness___portability_': 'satisfaction_compactness',
}

# Importance matrica - long common prefix mapped with sufix
IMPORTANCE_SUFFIX_MAP = {
    'Breathability___comfort_in_warm_weather_': 'importance_breathability',
    'Ease_of_use___putting_on_and_taking_off_': 'importance_ease_of_use',
    'Comfort_for_you__back__shoulders__': 'importance_comfort_parent',
    'Comfort_for_your_child__knees__hips__': 'importance_comfort_child',
    'Compactness___portability_': 'importance_compactness',
    'Price_': 'importance_price',
    'Safety_certifications_': 'importance_safety_certifications',
    'Warranty_': 'importance_warranty',
    'Brand_reputation_': 'importance_brand_reputation',
    'Design___aesthetics_': 'importance_design',
    'Material_quality_': 'importance_material_quality',
}

for col in df_raw.columns:
    if col.startswith('What_metters_to_you_in_a_carrier'):
        for suffix, short_name in IMPORTANCE_SUFFIX_MAP.items():
            if col.endswith(suffix):
                RENAME_MAP[col] = short_name
                break
df = df_raw.rename(columns=RENAME_MAP)

In [ ]:
def coalesce_columns(df, cols, new_name, drop_originals=True):
    existing = [c for c in cols if c in df.columns]
    df[new_name] = df[existing].bfill(axis=1).iloc[:, 0]
    if drop_originals:
        df.drop(columns=existing, inplace=True)
    return df

# 1) Would you try
WOULD_TRY_COLS = [
    'Would_you_be_willing_to_try_a_hip_sling_carrier_that_looks_like_the_one_in_the_picture_',
    'Would_you_be_willing_to_try_a_new_hip_sling_baby_carrier_that_looks_like_the_one_in_the_picture_',
]
df = coalesce_columns(df, cols=WOULD_TRY_COLS, new_name='would_try_hip_sling')

# 2) Willingness to pay
WTP_COLS = [
    'How_much_would_you_be_willing_to_pay_for_this_hip_sling_carrier___in_euros__1',
    'How_much_would_you_be_willing_to_pay_for_a_hip_baby_carrier___in_euros__1',
    'How_much_would_you_be_willing_to_pay_for_this_hip_sling_carrier___in_euros_',
]
df = coalesce_columns(df, WTP_COLS, 'wtp_eur')

# 3) Interested in launch notification
INTERESTED_COLS = [
    'Would_you_be_interested_in_being_the_first_to_know_when_this_carrier_launches__1',
    'Would_you_be_interested_in_being_the_first_to_know_when_this_carrier_launches__2',
    'Would_you_be_interested_in_being_the_first_to_know_when_this_carrier_launches_',
]
df = coalesce_columns(df, INTERESTED_COLS, 'interested_in_launch')

# 4) Legacy importance kolone -> spoji sa novim matrix kolonama
LEGACY_IMPORTANCE_MAP = {
    'Questions_and_rating__How_important_is_it_that_the_baby_carrier_is_breathable_and_comfortable_in_summer__': 'importance_breathability',
    'Questions_and_rating__How_important_is_ease_of_use_and_folding__': 'importance_ease_of_use',
    'Questions_and_rating__How_important_is_premium__high_quality_material__': 'importance_material_quality',
}
for legacy_col, target_col in LEGACY_IMPORTANCE_MAP.items():
    if legacy_col in df.columns:
        df[target_col] = df[target_col].fillna(df[legacy_col])
        df.drop(columns=[legacy_col], inplace=True)

/tmp/ipykernel_1806/1157845510.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[new_name] = df[existing].bfill(axis=1).iloc[:, 0]


In [ ]:
# Cleanup
df = df.rename(columns=RENAME_MAP)
df['outdoor_time_per_day_hrs'] = df['outdoor_time_per_day_hrs'].str.replace(
    r'\s*hours?\s*', '', regex=True
).str.strip()

# Additional data normalization for 'interested_in_launch'
df['interested_in_launch'] = df['interested_in_launch'].replace({True: 'Yes', False: 'No'}).fillna('No')

# Fill missing values for 'would_try_hip_sling'
df['would_try_hip_sling'] = df['would_try_hip_sling'].replace({'None': 'Maybe'}).fillna('Maybe')


In [ ]:
# Dummy columns creation

# Split transport type, family_activities and switch_reasons
MULTISELECT_COLS = ['transport_type', 'family_activities', 'switch_reasons']

df['respondent_id'] = df.index

for col in MULTISELECT_COLS:
    if col not in df.columns:
        continue

    # Creating dummy variables for multi-select columns
    dummies = df[col].str.get_dummies(sep=', ')
    dummies = dummies.add_prefix(f'{col}_')
    df = pd.concat([df, dummies], axis=1)

    df.drop(columns=[col], inplace=True)


In [ ]:
# Map missing state and city values

geolocator = Nominatim(user_agent="nomad_survey_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)  # OSM - max 1 req/sec

def parse_location_geopy(raw):
    if pd.isna(raw) or not str(raw).strip():
        return pd.Series({'city': None, 'state': None})

    try:
        location = geocode(str(raw), language='en', addressdetails=True)
        if location and 'address' in location.raw:
            addr = location.raw['address']
            city = addr.get('city') or addr.get('town') or addr.get('village') or addr.get('municipality')
            state = addr.get('country')
            return pd.Series({'city': city, 'state': state})
    except Exception:
        pass

    return pd.Series({'city': None, 'state': None})

# Cache - do not call 2 times for the same entry
unique_locations = df['location_raw'].dropna().unique()
location_cache = {}
for loc in unique_locations:
    location_cache[loc] = parse_location_geopy(loc)

df[['city', 'state']] = df['location_raw'].apply(
    lambda x: location_cache.get(x, pd.Series({'city': None, 'state': None}))
)
df.drop(columns=['location_raw'], inplace=True)

In [ ]:
def clean_city_name(city):
    if pd.isna(city):
        return city
    city = re.sub(r'^City of\s+', '', city, flags=re.IGNORECASE)
    return city.strip()

df['city'] = df['city'].apply(clean_city_name)
df[['city', 'state']].drop_duplicates().sort_values('state')

,city,state
21,Vienna,Austria
47,Zagreb,Croatia
41,Limassol,Cyprus
69,Heidelberg,Germany
52,Tivat,Montenegro
3,None,Serbia
4,Belgrade,Serbia
39,Zajecar,Serbia
42,Niš,Serbia
0,Valencia,Spain


In [ ]:
# Clan clumn naming of special characters, spaces etc for proper DWH entry

def clean_column_name(col):
    col = col.strip().lower()
    col = re.sub(r'[^a-z0-9_]+', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

df.columns = [clean_column_name(c) for c in df.columns]

In [ ]:
# Write clean survey data to intermediate dataset ready for analysis

df['updated_at'] = datetime.now(timezone.utc)

DATASET_INTERMEDIATE = "intermediate"
TABLE_CLEAN = "int_clean_survey_responses"

table_id = f"{PROJECT_ID}.{DATASET_INTERMEDIATE}.{TABLE_CLEAN}"

job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")

job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
job.result()

print(f"{df.shape[0]} rows written in {table_id} in {df['updated_at'].iloc[0]}")

71 rows written in nomad-503212.intermediate.int_clean_survey_responses in 2026-08-04 11:41:22.442277+00:00
